# Sekilas Jaringan Saraf Pertama: Klasifikasi Digit MNIST

**Buku:** *Deep Learning dengan Python* — Bab 2, Blok Bangunan Matematis

Kita membangun, melatih, dan menguji jaringan saraf dua lapisan untuk mengenali angka tulisan tangan. Jangan khawatir jika belum semua baris dipahami; setiap bagian akan dibedah pada sisa bab.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- `input_shape=` pada lapisan diganti `keras.Input(...)` di awal `Sequential`.
- Label tidak lagi di-*one-hot*; memakai `sparse_categorical_crossentropy` (lebih hemat memori).
- Akurasi uji dicetak dengan format persen; ditambah prediksi satu contoh.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Memuat data
MNIST tersedia langsung di Keras sebagai empat larik NumPy.

In [ ]:
from keras.datasets import mnist

(gambar_latih, label_latih), (gambar_uji, label_uji) = mnist.load_data()
print("Latih:", gambar_latih.shape, label_latih.shape)
print("Uji  :", gambar_uji.shape, label_uji.shape)
print("10 label pertama:", label_latih[:10])

## 2. Arsitektur jaringan
Dua lapisan `Dense`: lapisan tersembunyi 512 unit (ReLU) dan lapisan keluaran 10 kelas (softmax).

In [ ]:
from keras import layers

model = keras.Sequential([
    keras.Input(shape=(28 * 28,)),
    layers.Dense(512, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
model.summary()

## 3. Kompilasi: pengoptimal, fungsi loss, metrik

In [ ]:
model.compile(optimizer="rmsprop",
              loss="sparse_categorical_crossentropy",   # label berupa bilangan bulat 0-9
              metrics=["accuracy"])

## 4. Prapemrosesan
Ubah citra `(60000, 28, 28)` bertipe `uint8` [0, 255] menjadi `(60000, 784)` bertipe `float32` [0, 1].

In [ ]:
x_latih = gambar_latih.reshape((60000, 28 * 28)).astype("float32") / 255
x_uji = gambar_uji.reshape((10000, 28 * 28)).astype("float32") / 255

## 5. Pelatihan

In [ ]:
riwayat = model.fit(x_latih, label_latih, epochs=5, batch_size=128)

## 6. Evaluasi pada data uji
Selisih antara akurasi latih dan uji adalah gejala awal *overfitting*.

In [ ]:
loss_uji, akurasi_uji = model.evaluate(x_uji, label_uji, verbose=0)
print(f"Akurasi uji: {akurasi_uji:.2%}")

prediksi = model.predict(x_uji[:1], verbose=0)
print("Prediksi kelas citra uji pertama:", prediksi.argmax(), "| label sebenarnya:", label_uji[0])